# V54 fork wins 72.5% head to head against V54

**Fork of [Kaggriculture V54, Productive Wheat and Patient Sale](https://www.kaggle.com/code/ahmedberatozer/kaggriculture-v54-productive-wheat-and-patient) by [Ahmed Berat Özer](https://www.kaggle.com/ahmedberatozer).**
V54 is the strongest public agent I found. In my arena it beat Thomas Tschinkel's v13, Master Engine V4 and The 2965 Master
Hybrid Engine in 48 of 48 paired games. This fork changes two constants and beats V54 head to head.

| run in this notebook, both seats, seeds 64000..64019 | games | fork W-L-T | fork winrate, Wilson 95% | mean margin |
|---|---|---|---|---|
| fork vs V54 | 40 | 26-8-6 | 0.725 [0.572, 0.839] | $381 |
| fork vs v13 | 40 | 34-6-0 | 0.850 [0.709, 0.929] | $484 |
| V54 vs v13 (reference) | 40 | 40-0-0 | 1.000 [0.912, 1.000] | $108 |

My machine, same harness, other seeds: fork vs V54 31-7-2 over 40 games on seeds 63000..63019, winrate 0.800 [0.652, 0.895], mean margin $188.

**The change.** V54 inherits v13's RACE layer: it sells a planned lot early when its tape plans to sell that lot within the
next 40 turns, and stretches the window to at most 48 once it sees the rival selling ahead of it. The fork starts at 72.
Against V54 or any other file built on that layer, the fork is the one that sells first. Cell 3 prints the diff.

**What it does not claim.** The edge is small, about $381 a game against V54, and it is not free: against the older
files of the same family the fork drops some games V54 wins (section 4). It is also one move in an arms race: a copy with a
longer window beats it the same way.

Runtime: about 20 minutes on Kaggle's 4 CPUs (the 120 arena games took 778 s in version 2, the sales ledger replays 40 more, and the last cell prints the total), CPU only. Internet is on for one pip install of the game engine. Winrate counts a tie as half.

## 1. Credits

- **Ahmed Berat Özer**, [V54](https://www.kaggle.com/code/ahmedberatozer/kaggriculture-v54-productive-wheat-and-patient): the agent. This notebook reads his `main.py` from the attached notebook output
  and checks the SHA-256 he published.
- The lineage V54 credits and keeps byte for byte in `main.py`: Tetsutani, haideptry, Dmitrii Gluzdov, prvsiyan, and
  through them Thomas Tschinkel, Yusuke Hayashi (yhay81), aurax7, destbreso and the other authors named in the header.
- **Thomas Tschinkel**, [The Metav4 Farm: Submission v13](https://www.kaggle.com/code/thomastschinkel/the-metav4-farm-submission-v13): the RACE layer this fork retunes, and the second
  opponent below. His file is read from its own notebook output the same way.
- Field opponents measured on my machine only: Guru Prasaath S (Master Engine V4), haideptry (The 2965 Master Hybrid Engine),
  Thomas Tschinkel's v9.

Licence: Apache-2.0, same as V54. The lines I add are marked with my name at the end of `main.py`.

In [ ]:
import hashlib, os
from pathlib import Path

WANT = {"v54_main": "5fbb75c9c40e6d9e26d95272ace47329b1ca319b3b2118232404de30806e7e9c", "v13_main": "9d63494603f88219857a3101d7dc19cc750ded04e5886732ee428580326967d9"}
found = {}
for root, _, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith(".py"):
            b = Path(root, f).read_bytes()
            for name, sha in WANT.items():
                if hashlib.sha256(b).hexdigest() == sha:
                    found[name] = b
for name, sha in WANT.items():
    if name not in found and Path(name + ".py").exists():  # local runs
        found[name] = Path(name + ".py").read_bytes()
    assert name in found and hashlib.sha256(found[name]).hexdigest() == sha, "missing " + name
    Path(name + ".py").write_bytes(found[name])
src = found["v54_main"]

LAYER = b"""

# busyaprime fork of V54, 2026-09-21. Apache-2.0 like everything above.
# The RACE layer (from Thomas Tschinkel's v9) pre-sells a planned lot when the tape plans it within V9_RACE_DEFAULT
# turns (40), and stretches that to V9_RACE_MAX (48) when it sees the rival sell ahead of it. The fork starts at 72.
# The cap of 96 never binds: the stretch is the rival's lead plus 12, and the layer searches 30 turns for a lead.
V9_RACE_DEFAULT = 72
V9_RACE_MAX = 96
"""
Path("main.py").write_bytes(src + LAYER)
fork = Path("main.py").read_bytes()
assert fork.startswith(src) and len(fork) == len(src) + len(LAYER)
print("V54 main.py", len(src), "bytes, sha256", WANT["v54_main"])
print("fork main.py", len(fork), "bytes, sha256", hashlib.sha256(fork).hexdigest())
print("fork = V54 byte for byte, plus these lines at the end:")
print(fork[len(src):].decode())

The Kaggle image ships kaggle-environments 1.29.3. Its kaggriculture is an older build of the game: on the first version of
this notebook every game ended with both banks at $26 or less and 106 of 120 were ties, which says nothing about the ladder. The
arena below runs on 1.32.7, the build I play locally, and the cell stops if it gets anything else.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle-environments==1.32.7"], check=True)

## 2. Paired arena

Each seed is played twice with the seats swapped, so a seat advantage cannot pass for skill. A seed where one agent wins
both seats is an edge on that map; a split is a tie. None of these seeds were used to pick 72: the screen that picked it ran
on seeds 61100..61111 (section 5).

In [ ]:
import contextlib, csv, io, math, time
import importlib.metadata as imd
import multiprocessing as mp
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    from kaggle_environments import make
    import kaggle_environments.envs.kaggriculture.kaggriculture as K
print("kaggle-environments", imd.version("kaggle-environments"), "| CPUs", os.cpu_count())
assert imd.version("kaggle-environments") == "1.32.7", "restart the kernel after the pip cell"

def play(job):
    seed, a0, a1 = job
    t = time.time()
    env = make("kaggriculture", configuration={"episodeSteps": 720, "seed": seed}, debug=False)
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        env.run([a0 + ".py", a1 + ".py"])
    last = env.steps[-1]
    return dict(seed=seed, agent0=a0, agent1=a1, bank0=last[0].reward, bank1=last[1].reward,
                status0=last[0].status, status1=last[1].status, secs=round(time.time() - t, 1))

SEED0, NSEEDS = 64000, 20
PAIRS = [("main", "v54_main"), ("main", "v13_main"), ("v54_main", "v13_main")]
jobs = [(s, p, q) for x, y in PAIRS for s in range(SEED0, SEED0 + NSEEDS) for p, q in ((x, y), (y, x))]
T0 = time.time()
with mp.get_context("fork").Pool(os.cpu_count()) as pool:
    rows = pool.map(play, jobs, chunksize=1)
print(len(rows), "games in", round(time.time() - T0), "s")
assert all(r["status0"] == "DONE" and r["status1"] == "DONE" for r in rows)
with open("arena.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)

In [ ]:
import pandas as pd

def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z * z / n; c = p + z * z / (2 * n); h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return (c - h) / d, (c + h) / d

NAMES = {"main": "fork", "v54_main": "V54", "v13_main": "v13"}
out = []
for x, y in PAIRS:
    m = {}
    for r in rows:
        if {r["agent0"], r["agent1"]} != {x, y}:
            continue
        d = r["bank0"] - r["bank1"] if r["agent0"] == x else r["bank1"] - r["bank0"]
        m.setdefault(r["seed"], []).append(d)
    ds = [d for v in m.values() for d in v]
    w, l, t = sum(d > 0 for d in ds), sum(d < 0 for d in ds), sum(d == 0 for d in ds)
    lo, hi = wilson(w + 0.5 * t, len(ds))
    both = sum(all(d > 0 for d in v) for v in m.values()); lost = sum(all(d < 0 for d in v) for v in m.values())
    out.append({"agent": NAMES[x], "opponent": NAMES[y], "games": len(ds), "W-L-T": f"{w}-{l}-{t}",
                "winrate": round((w + 0.5 * t) / len(ds), 3), "wilson95": f"[{lo:.3f}, {hi:.3f}]",
                "mean margin $": round(sum(ds) / len(ds)),
                "seeds won both / split / lost both": f"{both}/{len(m) - both - lost}/{lost}"})
table = pd.DataFrame(out)
table

## 3. Where the extra money comes from

The engine fills both players' SELL orders one unit at a time, and every unit sold pushes the price down for the next one.
When both farms hold the same lot, whoever sells it first gets the better part of the price curve. The RACE layer knows
this: it watches the market, works out how far ahead of its own tape the rival sold, and moves its own sales forward to
match, up to 48 turns. Two copies of it settle somewhere between 40 and 48 turns. The fork starts at 72, so on a lot both
farms plan to sell, it gets to the market first.

The cell below replays the fork against V54 on all twenty seeds, both seats, logs every unit each side sold, and compares
what each got per unit.

In [ ]:
def ledger_game(job):
    seed, seats = job
    log, ctx = [], {}
    _commit, _market = K._commit_unit, K._process_market
    def market_logged(state, env):
        ctx["farms"] = state[0].observation.farms
        return _market(state, env)
    def commit_logged(op, item, price, farm, *rest):
        ok = _commit(op, item, price, farm, *rest)
        if ok and op == "SELL":
            seat = next(i for i, f in enumerate(ctx["farms"]) if f is farm)
            log.append((seed, seats[seat], item, price))
        return ok
    K._process_market, K._commit_unit = market_logged, commit_logged
    try:
        env = make("kaggriculture", configuration={"episodeSteps": 720, "seed": seed}, debug=False)
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            env.run([x + ".py" for x in seats])
    finally:
        K._process_market, K._commit_unit = _market, _commit
    return log

ljobs = [(s, seats) for s in range(SEED0, SEED0 + NSEEDS) for seats in (("main", "v54_main"), ("v54_main", "main"))]
with mp.get_context("fork").Pool(os.cpu_count()) as pool:
    LOG = [x for part in pool.map(ledger_game, ljobs, chunksize=1) for x in part]
sold = pd.DataFrame(LOG, columns=["seed", "agent", "item", "price"]).replace({"agent": NAMES})
per = sold.groupby(["item", "agent"])["price"].agg(units="count", revenue="sum", avg_price="mean").round(1).unstack("agent")
per[("revenue", "fork - V54")] = per[("revenue", "fork")] - per[("revenue", "V54")]
print("sales revenue, fork minus V54, over", len(ljobs), "games:", int(per[("revenue", "fork - V54")].sum()))
per

Over these 40 games the fork's sales bring in $15,250 more than V54's, which is the whole mean margin of section 2
($381 a game). Strawberries carry $12,180 of it and milk $3,184: the same number of units, sold at a slightly better
average price (121.7 against 120.5 for strawberries). V54 does a little better on wool. Nothing else moves. So the edge
is not a different farm, it is the same farm getting to the strawberry and milk buyers first, a dollar a unit at a time.

## 4. Against the rest of the field

A longer pre-sale window could cost money against an agent that does not share the schedule: selling 72 turns early means
selling before the town has drained the market. I checked this on my machine with the same paired harness, seeds
62000..62007, both seats, 16 games per row. The field files are
about 1 MB each and are not reproduced here.

| opponent | V54: W-L-T | V54: winrate | V54: mean margin | fork: W-L-T | fork: winrate | fork: mean margin |
|---|---|---|---|---|---|---|
| Master Engine V4 (Guru Prasaath S) | 16-0-0 | 1.000 | $28 | 12-4-0 | 0.750 | -$6 |
| The 2965 Master Hybrid Engine (haideptry) | 16-0-0 | 1.000 | $57 | 12-4-0 | 0.750 | $23 |
| v13 (Thomas Tschinkel) | 16-0-0 | 1.000 | $106 | 12-4-0 | 0.750 | $72 |
| v9 (Thomas Tschinkel) | 14-2-0 | 0.875 | $1,200 | 14-2-0 | 0.875 | $1,486 |
| v13 with the same 72 window (my other fork) | 10-6-0 | 0.625 | $140 | 16-0-0 | 1.000 | $106 |

Against Master Engine V4, The 2965 and v13 together V54 won 48 of 48 and the fork won 36 of 48. The fork gives up games V54 wins. Those losses come from whole seeds lost in both seats, so they are maps where selling 72 turns early is
simply the wrong call, not seat noise. A 56-turn window does not buy the games back: on the same seeds it went 12-4-0 against Master Engine V4, 12-4-0 against The 2965, 12-4-0 against v13. The trade is: a clear win over V54 and over copies of it, for a few games against the
files one step behind it. Which side of that trade is right depends on who you meet on the ladder, and I have not measured
that. Eight seeds per row is thin; treat this table as a warning, not a rate.

## 5. What I tried and dropped

First on v13 itself, seeds 61100..61111, both seats, 24 games per row:

| variant of v13 | W-L-T vs v13 | winrate | Wilson 95% | mean margin |
|---|---|---|---|---|
| window 32, cap 48 | 12-8-4 | 0.583 | [0.388, 0.755] | $93 |
| window 48, cap 56 | 16-2-6 | 0.792 | [0.595, 0.908] | $123 |
| window 56, cap 64 | 18-2-4 | 0.833 | [0.641, 0.933] | $124 |
| window 72, cap 96 (kept) | 20-2-2 | 0.875 | [0.690, 0.957] | $132 |
| window 96, cap 96 | 20-2-2 | 0.875 | [0.690, 0.957] | $132 |
| window 144, cap 144 | 20-2-2 | 0.875 | [0.690, 0.957] | $132 |
| window 240, cap 240 | 20-2-2 | 0.875 | [0.690, 0.957] | $132 |
| v13 window, early-sale price gate at base - 10 | 2-12-10 | 0.292 | [0.149, 0.492] | $13 |
| v13 window, early-sale price gate at base + 20 | 2-20-2 | 0.125 | [0.043, 0.310] | -$231 |

The curve rises up to 72 and is flat after it: 96, 144 and 240 end 22 of these 24 games with exactly the same two banks as
72 and have the same record. My guess, not checked move by move, is that past 72 the tape's own pickups and purchases end
the look-ahead before the window does. Moving the price gate on
early sales (RACEPX margin) loses in both directions. The same 72 on V54 went 20-2-2 (0.875) on the same seeds.

The obvious limit: if anyone raises the window past 72, the edge goes to whoever looks further, and nothing here measures
that match.

In [ ]:
import gzip, tarfile
data = open("main.py", "rb").read()
with open("submission.tar.gz", "wb") as raw:
    with gzip.GzipFile(fileobj=raw, mode="wb", mtime=0) as z:
        with tarfile.open(fileobj=z, mode="w", format=tarfile.GNU_FORMAT) as tar:
            info = tarfile.TarInfo("main.py"); info.size = len(data); info.mtime = 0; info.mode = 0o644
            tar.addfile(info, io.BytesIO(data))
for f in ("v54_main.py", "v13_main.py"):
    os.remove(f)
print("submission.tar.gz written, main.py sha256", hashlib.sha256(data).hexdigest())
print("whole notebook:", round(time.time() - T0), "s after the arena started")